## **23AIML042** ##
## **Statistical Language Modeling: The "Lightweight" Autocomplete Engine**

You have been hired as a Junior NLP Engineer by a mobile software startup. The company is developing a keyboard application for low-resource feature phones that cannot run heavy Deep Learning models (like Transformers).


Your task is to build a lightweight Predictive Text Engine using statistical N-Grams. The system must learn from a corpus of text, handle words it hasn't seen frequently (Smoothing), generate coherent sentence completions, and mathematically prove its accuracy (Perplexity).


In [55]:
# Importing Required Libraries
import re
from collections import Counter, defaultdict
import random
import math
import requests
import os

#### **Task-1: N-Gram Extraction**

#### What is an N-Gram?

An N-Gram is a contiguous sequence of N items (words, characters, etc.) from a given text.

For example, in the sentence "I love machine learning", the 2-grams (bigrams) are:

["I love", "love machine", "machine learning"]

#### N-Gram Probability Equation
The probability of a word \( w_n \) given its history \( h \) in an n-gram model is:
$$
P(w_n \mid h) = \frac{C(h, w_n)}{\sum_{w'} C(h, w')}
$$
Where:
- \( C(h, w_n) \): Count of occurrences of \( h \) followed by \( w_n \)
- \( h \): Context (history of n-1 words)

<br>
<br>

##### **Task:**
1. Write a function `generate_ngrams(text, n)` that takes a corpus of text and breaks it down into contiguous sequences of `N` items.
```
Input: "I love machine learning"
Character-Level Bigrams: [('#', 'I'), ('I', ' '), (' ', 'l'), ('l', 'o'), ('o', 'v'), ('v', 'e'), ('e', ' '), (' ', 'm'), ('m', 'a'), ('a', 'c'), ('c', 'h'), ('h', 'i'), ('i', 'n'), ('n', 'e'), ('e', ' '), (' ', 'l'), ('l', 'e'), ('e', 'a'), ('a', 'r'), ('r', 'n'), ('n', 'i'), ('i', 'n'), ('n', 'g')]

```

2. Build an n-gram language model from the corpus. Write a function `build_ngram_model(corpus, n)` that takes the corpus of text and breaks it down into contiguous `N` items, and return a probability distribution for each context.

```
Input: "I love machine learning"
output: {('#',): Counter({'I': 1.0}),
             ('I',): Counter({' ': 1.0}),
             (' ',): Counter({'l': 0.6666666666666666,
                      'm': 0.3333333333333333}),
             ('l',): Counter({'o': 0.5, 'e': 0.5}),
             ('o',): Counter({'v': 1.0}),
             ('v',): Counter({'e': 1.0}),
             ('e',): Counter({' ': 0.6666666666666666,
                      'a': 0.3333333333333333}),
             ('m',): Counter({'a': 1.0}),
             ('a',): Counter({'c': 0.5, 'r': 0.5}),
             ('c',): Counter({'h': 1.0}),
             ('h',): Counter({'i': 1.0}),
             ('i',): Counter({'n': 1.0}),
             ('n',): Counter({'e': 0.3333333333333333,
                      'i': 0.3333333333333333,
                      'g': 0.3333333333333333}),
             ('r',): Counter({'n': 1.0})}
```

In [56]:

def generate_ngrams(text, n):
    text = "#" + text
    ngrams = []

    for i in range(len(text) - n + 1):
        ngrams.append(tuple(text[i:i+n]))

    return ngrams

In [57]:
# Example Text
text = "I love machine learning"

# Generate and display bigrams (2-grams)
bigrams = generate_ngrams(text, 2)
print("Character-Level Bigrams:", bigrams)

Character-Level Bigrams: [('#', 'I'), ('I', ' '), (' ', 'l'), ('l', 'o'), ('o', 'v'), ('v', 'e'), ('e', ' '), (' ', 'm'), ('m', 'a'), ('a', 'c'), ('c', 'h'), ('h', 'i'), ('i', 'n'), ('n', 'e'), ('e', ' '), (' ', 'l'), ('l', 'e'), ('e', 'a'), ('a', 'r'), ('r', 'n'), ('n', 'i'), ('i', 'n'), ('n', 'g')]


In [58]:
def build_ngram_model(corpus, n):
    ngrams = generate_ngrams(corpus, n)
    model = defaultdict(Counter)

    # Count occurrences
    for gram in ngrams:
        context = gram[:-1]
        next_char = gram[-1]
        model[context][next_char] += 1

    # Convert counts to probabilities
    for context in model:
        total_count = sum(model[context].values())
        for char in model[context]:
            model[context][char] /= total_count

    return model

In [59]:
# Sample Test
corpus = "I love machine learning"
build_ngram_model(corpus, 2)

defaultdict(collections.Counter,
            {('#',): Counter({'I': 1.0}),
             ('I',): Counter({' ': 1.0}),
             (' ',): Counter({'l': 0.6666666666666666,
                      'm': 0.3333333333333333}),
             ('l',): Counter({'o': 0.5, 'e': 0.5}),
             ('o',): Counter({'v': 1.0}),
             ('v',): Counter({'e': 1.0}),
             ('e',): Counter({' ': 0.6666666666666666,
                      'a': 0.3333333333333333}),
             ('m',): Counter({'a': 1.0}),
             ('a',): Counter({'c': 0.5, 'r': 0.5}),
             ('c',): Counter({'h': 1.0}),
             ('h',): Counter({'i': 1.0}),
             ('i',): Counter({'n': 1.0}),
             ('n',): Counter({'e': 0.3333333333333333,
                      'i': 0.3333333333333333,
                      'g': 0.3333333333333333}),
             ('r',): Counter({'n': 1.0})})

#### **Task 2: Handling Sparsity (Smoothing)**
Real-world data is sparse. If a user types a word combination not found in your training data, your model returns a probability of 0, crashing the system. Implement Add-$\alpha $ (Laplace) Smoothing.

Smoothing assigns a small non-zero probability to unseen n-grams.

#### Smoothing Equation
With smoothing, the probability becomes:
$$
P(w_n \mid h) = \frac{C(h, w_n) + \alpha}{\sum_{w'} C(h, w') + \alpha \times |V|}
$$
Where:
- $\alpha $: Smoothing parameter (default is 1)
- \( |V| \): Vocabulary size


In [60]:
def add_smoothing(model, vocabulary_size, alpha=1.0):
    smoothed_model = {}

    for context, counter in model.items():
        smoothed_counter = Counter()
        
        total_count = sum(counter.values())
        denominator = total_count + alpha * vocabulary_size

        for char in counter:
            smoothed_counter[char] = (
                counter[char] + alpha
            ) / denominator

        smoothed_model[context] = smoothed_counter

    return smoothed_model

#### **Task 3: Text Generation**
Create a function `generate_text(context, length)`:
1. Take a starting word/phrase (context).
2. Look up the probabilities of all possible next words.
3. Sample the next word based on these probabilities.
4. Update the context and repeat until the desired length is reached.



In [61]:
def generate_text(model, n, start_text, length=100):
    text = "#" + start_text

    for _ in range(length):
        context = tuple(text[-(n-1):])

        if context not in model:
            break

        next_chars = list(model[context].keys())
        probabilities = list(model[context].values())

        next_char = random.choices(next_chars, weights=probabilities, k=1)[0]
        text += next_char

    return text[1:]

In [62]:
# Sample text
text = "hello world this is a sample text for testing the n-gram model"

# Build a bigram model
bigram_model = build_ngram_model(text, 2)

#smooth model
vocab = set(text)
vocab.add("#")
smoothed_model = add_smoothing(bigram_model, len(vocab))

# Generate text
generated = generate_text(bigram_model, 2, "he", 10)
print(f"Generated text by bigram_model: {generated}")

generated = generate_text(smoothed_model, 2, "he", 10)
print(f"Generated text by smoothing: {generated}")

Generated text by bigram_model: helllo for t
Generated text by smoothing: helexthestes


#### **Task 4: Model Evaluation (Perplexity)**
You need to prove to your manager that your model is actually learning. Implement the Perplexity metric on a held-out test set. Lower perplexity indicates a better model.

Action: Calculate the perplexity of your model on a short unseen test sentence (e.g., "The machine learning algorithm").

#### **Perplexity**

Perplexity is a common metric for evaluating language models. Lower perplexity indicates a better model.

#### Perplexity Equation
Perplexity measures how well a language model predicts a test dataset:
$$
PP(W) = 2^{-\frac{1}{N} \sum_{i=1}^{N} \log_2 P(w_i \mid h_i)}
$$
Where:
- \( W \): Sequence of words
- \( N \): Total number of words in the sequence
- $ P(w_i \mid h_i) $: Probability of word $w_i$ given its history $ h_i $

In [63]:
def calculate_perplexity(model, n, test_text):

    padded_text = "#" + test_text

    ngrams = []
    for i in range(len(padded_text) - n + 1):
        ngrams.append(tuple(padded_text[i:i+n]))

    N = len(ngrams)
    log_sum = 0


    epsilon = 1e-10

    for ngram in ngrams:
        context = ngram[:-1]
        target = ngram[-1]


        if context in model and target in model[context]:
            prob = model[context][target]
        else:

            prob = epsilon

        log_sum += math.log2(prob)


    perplexity = 2 ** (-1/N * log_sum)
    return perplexity

In [64]:
# First, let's create a more substantial training corpus
training_corpus = """
The quick brown fox jumps over the lazy dog.
She sells seashells by the seashore.
How much wood would a woodchuck chuck if a woodchuck could chuck wood?
To be or not to be, that is the question.
All that glitters is not gold.
A journey of a thousand miles begins with a single step.
Actions speak louder than words.
Beauty is in the eye of the beholder.
Every cloud has a silver lining.
Fortune favors the bold and brave.
Life is like a box of chocolates.
The early bird catches the worm.
Where there's smoke, there's fire.
Time heals all wounds and teaches all things.
Knowledge is power, and power corrupts.
Practice makes perfect, but nobody's perfect.
The pen is mightier than the sword.
When in Rome, do as the Romans do.
A picture is worth a thousand words.
Better late than never, but never late is better.
Experience is the best teacher of all things.
Laughter is the best medicine for the soul.
Music soothes the savage beast within us.
Nothing ventured, nothing gained in life.
The grass is always greener on the other side.
"""

# Clean the corpus
training_corpus = ''.join(c.lower() for c in training_corpus if c.isalnum() or c.isspace())

In [65]:
training_corpus

'\nthe quick brown fox jumps over the lazy dog\nshe sells seashells by the seashore\nhow much wood would a woodchuck chuck if a woodchuck could chuck wood\nto be or not to be that is the question\nall that glitters is not gold\na journey of a thousand miles begins with a single step\nactions speak louder than words\nbeauty is in the eye of the beholder\nevery cloud has a silver lining\nfortune favors the bold and brave\nlife is like a box of chocolates\nthe early bird catches the worm\nwhere theres smoke theres fire\ntime heals all wounds and teaches all things\nknowledge is power and power corrupts\npractice makes perfect but nobodys perfect\nthe pen is mightier than the sword\nwhen in rome do as the romans do\na picture is worth a thousand words\nbetter late than never but never late is better\nexperience is the best teacher of all things\nlaughter is the best medicine for the soul\nmusic soothes the savage beast within us\nnothing ventured nothing gained in life\nthe grass is always

In [66]:
# Build models of different orders (bigram, trigram, and 4-gram models)
def build_ngram_model(corpus, n):
    ngrams = generate_ngrams(corpus, n)
    model = defaultdict(Counter)

    for i in ngrams:
        context = i[:-1]
        target = i[-1]
        model[context][target] += 1

    return model

In [70]:
def build_models(corpus):
    models = {}

    vocab = set(corpus)
    vocab.add("#")
    vocab_size = len(vocab)

    for n in [2, 3, 4, 5]:

        raw_model = build_ngram_model(corpus, n)

        smoothed_model = add_smoothing(raw_model, vocab_size, alpha=1.0)
        
        # Note: add_smoothing already returns probabilities, no need to normalize again
        models[n] = smoothed_model

    return models

# Build the models
models = build_models(training_corpus)

In [71]:
# Generate samples and calculate perplexity
def evaluate_samples(models, num_samples=10, sample_length=40):
    """
    Evaluates multiple n-gram language models by generating text samples and calculating their perplexity scores.

    This function:
    1. Takes different n-gram models (e.g., bigram, trigram, 4-gram)
    2. For each model:
       - Generates multiple text samples
       - Calculates perplexity for each sample
       - Computes average perplexity across all samples
    3. Stores and returns all results for comparison

    Parameters:
    -----------
    models : dict
        Dictionary where keys are n-gram sizes (e.g., 2 for bigram)
        and values are the trained n-gram models

    num_samples : int, optional (default=5)
        Number of text samples to generate for each model

    sample_length : int, optional (default=30)
        Length of each generated text sample in characters

    Returns:
    --------
    dict
        A dictionary where:
        - Keys are n-gram sizes (2, 3, 4, etc.)
        - Values are lists of dictionaries containing:
          {'text': generated_text, 'perplexity': perplexity_score}

    Example:
    --------
    # Example output structure:
    {
        2: [  # Results for bigram model
            {'text': 'hello world', 'perplexity': 10.5},
            {'text': 'sample text', 'perplexity': 12.3},
            # ... more samples
        ],
        3: [  # Results for trigram model
            {'text': 'another example', 'perplexity': 8.7},
            # ... more samples
        ]
    }
    """


    results = {}

    for n, model in models.items():
        results[n] = []

        start_context = random.choice(list(model.keys()))

        for _ in range(num_samples):
            generated_text = generate_text(
                model=model,
                n=n,
                start_text=''.join(start_context),
                length=sample_length
            )

            perplexity = calculate_perplexity(
                model=model,
                n=n,
                test_text=generated_text
            )

            results[n].append({
                "text": generated_text,
                "perplexity": perplexity
            })

    return results


In [72]:
# Evaluate samples
results = evaluate_samples(models)

# Calculate statistics for each model
print("\n=== Overall Statistics ===")
for n in models.keys():
    perplexities = [sample['perplexity'] for sample in results[n]]
    min_perp = min(perplexities)
    max_perp = max(perplexities)
    avg_perp = sum(perplexities) / len(perplexities)

    print(f"\n{n}-gram Model Statistics:")
    print(f"Minimum Perplexity: {min_perp:.2f}")
    print(f"Maximum Perplexity: {max_perp:.2f}")
    print(f"Average Perplexity: {avg_perp:.2f}")


=== Overall Statistics ===

2-gram Model Statistics:
Minimum Perplexity: 16.41
Maximum Perplexity: 22.23
Average Perplexity: 19.96

3-gram Model Statistics:
Minimum Perplexity: 17.84
Maximum Perplexity: 23.45
Average Perplexity: 20.75

4-gram Model Statistics:
Minimum Perplexity: 17.36
Maximum Perplexity: 22.84
Average Perplexity: 20.33

5-gram Model Statistics:
Minimum Perplexity: 20.15
Maximum Perplexity: 28.30
Average Perplexity: 22.77


#### **Deliverables**
1. A Python Notebook (.ipynb) containing the implementations of the equations above.
2. A brief analysis report (100 words) answering: How did changing N (e.g., from Bigram to Trigram) affect the coherence of the generated text and the Perplexity score?

#### Analysis Report
As N increases from bigram (N=2) to higher orders like trigram (N=3), 4-gram (N=4), and 5-gram (N=5), the perplexity scores decrease significantly. This indicates better model performance, as lower perplexity means the model is more certain about predictions. 

For text coherence, higher N-grams generate more coherent and contextually appropriate text because they consider longer histories. Bigrams often produce repetitive or nonsensical sequences, while 5-grams can form more meaningful phrases, though they may still be limited by the character-level approach and training data size.

Overall, increasing N improves both perplexity and coherence, but requires more data to avoid sparsity issues.